In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
SPECTROMETER_FREQUENCY_MHZ = 600.0
CSV_FILE_PATHS = [Path("data/mnova_data/substraty_peak-picking.csv")]#sorted(Path("data/mnova_data").glob("*2.csv"))
OUTPUT_DIR = Path("data/processed_spectra")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def parse_file(file_path):
    lines = Path(file_path).read_text().splitlines()[1:] # skip header

    peaks_data = []
    for line in lines:
        line_parts = line.split()
        # line_data = {
        #     "name": line_parts[1],
        #     "from_ppm": float(line_parts[2]),
        #     "to_ppm": float(line_parts[3]),
        #     "error": float(line_parts[4]),
        # }
        peaks_data.extend([
            {
                "multiplet_name": line_parts[1],
                "peak_in_multiplet": int(line_parts[i]),
                "position_ppm": float(line_parts[i + 1]),
                "height": float(line_parts[i + 2]),
                "width_hz": float(line_parts[i + 3]),
                "L/G": float(line_parts[i + 4]),
                "area": float(line_parts[i + 5]),
                }
            for i in range(5, len(line_parts), 6)
        ])

    # for peak_data in peaks_data:
    #     l_g = peak_data.pop("L/G")
    #     # print(peak_data["name"], l_g)
    #     peak_data["gaussian_fraction"] = np.clip(1 / (1 + np.clip(l_g, 0, None)), 0.0, 1.0)
        
    return peaks_data

def squeeze_peaks_data(df, position_start, maximal_gap_hz):
    df = df.copy().sort_values("position_hz").reset_index(drop=True)
    current_position = position_start
    new_positions = []
    for _, row in df.iterrows():
        if row["position_hz"] - current_position > maximal_gap_hz:
            current_position += maximal_gap_hz
        else:
            current_position = row["position_hz"]
        new_positions.append(current_position)
    df["position_hz"] = new_positions
    return df


In [6]:
for csv_file_path in CSV_FILE_PATHS:
    peaks_data = parse_file(csv_file_path)
    output_file_path = OUTPUT_DIR / (Path(csv_file_path).stem + ".csv")
    peaks_data = pd.DataFrame(peaks_data)
    peaks_data["position_hz"] = peaks_data["position_ppm"] * SPECTROMETER_FREQUENCY_MHZ
    peaks_data["gaussian_fraction"] = np.clip(1 / (1 + np.clip(peaks_data["L/G"], 0, None)), 0.0, 1.0)
    peaks_data = peaks_data.drop(columns=["L/G"])
    # peaks_data.to_csv(output_file_path, index=False)
    break

In [7]:
peaks_data

,multiplet_name,peak_in_multiplet,position_ppm,height,width_hz,area,position_hz,gaussian_fraction
0,M1,1,8.5735,0.89,0.46,7.06,5144.10,0.418410
1,M1,2,8.5706,1.29,1.05,25.57,5142.36,0.531915
2,M2,1,7.8076,0.92,0.50,7.81,4684.56,0.404858
3,M2,2,7.8052,3.33,0.98,60.45,4683.12,0.507614
4,M2,3,7.8028,2.93,1.12,69.12,4681.68,0.892857
...,...,...,...,...,...,...,...,...
80,M15,1,0.8296,0.23,1.58,6.60,497.76,0.465116
81,M15,2,0.8190,0.77,1.89,29.36,491.40,0.680272
82,M15,3,0.8112,0.35,1.31,7.72,486.72,0.380228
83,M15,4,0.8069,0.32,1.32,6.65,484.14,0.333333


In [8]:
position_start = 0.0
maximal_gap_hz = 10.0

In [16]:
squeezed_data = squeeze_peaks_data(peaks_data, position_start, maximal_gap_hz)
squeezed_data

,multiplet_name,peak_in_multiplet,position_ppm,height,width_hz,area,position_hz,gaussian_fraction
0,M15,5,0.7998,0.21,1.63,6.75,10.0,0.699301
1,M15,4,0.8069,0.32,1.32,6.65,20.0,0.333333
2,M15,3,0.8112,0.35,1.31,7.72,30.0,0.380228
3,M15,2,0.8190,0.77,1.89,29.36,40.0,0.680272
4,M15,1,0.8296,0.23,1.58,6.60,50.0,0.465116
...,...,...,...,...,...,...,...,...
80,M2,3,7.8028,2.93,1.12,69.12,810.0,0.892857
81,M2,2,7.8052,3.33,0.98,60.45,820.0,0.507614
82,M2,1,7.8076,0.92,0.50,7.81,830.0,0.404858
83,M1,2,8.5706,1.29,1.05,25.57,840.0,0.531915


In [17]:
peaks_data["position_hz"].max(), squeezed_data["position_hz"].max()

(np.float64(5144.099999999999), np.float64(850.0))

In [ ]:
peaks_data.sort_values("position_hz", inplace=True)
peaks_data.reset_index(drop=True, inplace=True)
peaks_data["position_hz"] -= peaks_data["position_hz"].min() - position_start
for i in range(1, len(peaks_data)):
    gap = peaks_data["position_hz"].iloc[i] - peaks_data["position_hz"].iloc[i - 1]
    print(gap)
    if gap > maximal_gap_hz:
        shift = gap - maximal_gap_hz
        peaks_data["position_hz"].iloc[i:] -= shift

4.259999999999991
2.580000000000041
4.67999999999995
6.360000000000014
10.0
10.0
10.0
1.9199999999998454
1.9199999999998454
1.9800000000002456
10.0
10.0
10.0
8.699999999999818
8.460000000000036
10.0
10.0
3.9599999999995816
10.0
4.2599999999997635
5.220000000000255
4.260000000000218
8.159999999999854
10.0
6.360000000000127
4.5
6.360000000000127
5.099999999999909
1.7399999999997817
10.0
6.420000000000073
3.1799999999998363
6.299999999999727
10.0
8.33999999999969
10.0
4.320000000000164
2.0399999999999636
4.320000000000164
10.0
4.619999999999891
6.360000000000127
4.739999999999782
10.0
10.0
10.0
10.0
10.0
10.0
1.319999999999709
1.5
2.0399999999999636
1.980000000000473
1.8599999999996726
3.180000000000291
2.8799999999991996
1.680000000000291
3.5999999999994543
3.660000000000764
1.8599999999996726
8.399999999999636
2.880000000000109
2.7600000000002183
1.680000000000291
5.460000000000036
2.519999999999527
4.380000000000109
2.5799999999999272
6.960000000000036
1.8599999999996726
5.880000000001